In [1]:
%pip install mlflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import mlflow
from mlflow import MlflowClient

print("MLflow version:", mlflow.__version__)

MLflow version: 3.17.2


In [2]:
# Data handling
import numpy as np
import pandas as pd

# Model building and evaluation (scikit-learn)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Experiment tracking
import mlflow
from mlflow import MlflowClient

print("MLflow version:", mlflow.__version__)

MLflow version: 3.17.2


In [3]:
# Fix the random seed so that everyone obtains the same data
np.random.seed(42)

number_of_orders = 500

# Create the input features
df = pd.DataFrame()
df["distance_km"] = np.random.uniform(1, 15, number_of_orders)
df["items"]       = np.random.randint(1, 10, number_of_orders)
df["hour"]        = np.random.randint(8, 23, number_of_orders)

# Create the target: base time + time for distance + time for items + noise
noise = np.random.normal(0, 3, number_of_orders)
df["delivery_minutes"] = 10 + 3 * df["distance_km"] + 1.5 * df["items"] + noise

# Separate the inputs (X) from the target (y)
X = df[["distance_km", "items", "hour"]]
y = df["delivery_minutes"]

# Split into 80% training data and 20% test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training rows:", len(X_train))
print("Test rows    :", len(X_test))
df.head()

Training rows: 400
Test rows    : 100


,distance_km,items,hour,delivery_minutes
0,6.243562,6,18,36.064045
1,14.310000,1,21,55.905380
2,11.247915,9,14,58.715004
3,9.381219,1,8,37.832316
4,3.184261,5,9,26.350811


In [4]:
for depth in [2, 5, 10]:
    # Train a decision tree with the chosen depth
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)

    # Evaluate it on the test set
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)

    print("max_depth =", depth, "   MAE =", round(mae, 2))

max_depth = 2    MAE = 4.59
max_depth = 5    MAE = 3.31
max_depth = 10    MAE = 4.07


In [5]:
# Store all MLflow records in a local SQLite database file
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# All runs in this notebook will belong to this experiment
mlflow.set_experiment("P06-delivery")

print("Tracking URI:", mlflow.get_tracking_uri())

2026/11/02 23:54:52 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/02 23:54:52 INFO mlflow.store.db.utils: Updating database tables
2026/11/02 23:55:00 INFO mlflow.tracking.fluent: Experiment with name 'P06-delivery' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow.db


In [6]:
with mlflow.start_run(run_name="first-tree"):

    depth = 4

    # Train and evaluate the model (same as before)
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    mae = mean_absolute_error(y_test, model.predict(X_test))

    # Record the parameters (the settings we chose)
    mlflow.log_param("max_depth", depth)
    mlflow.log_param("test_size", 0.2)

    # Record the metric (the result we obtained)
    mlflow.log_metric("mae_minutes", mae)

    # Record a tag (a label of our choice)
    mlflow.set_tag("student", "demo")

print("Run recorded. MAE =", round(mae, 2))

Run recorded. MAE = 3.47


In [7]:
# Read all runs of the experiment from the database
runs = mlflow.search_runs(experiment_names=["P06-delivery"])

# Display only the columns we are interested in
runs[["tags.mlflow.runName", "params.max_depth", "metrics.mae_minutes"]]

,tags.mlflow.runName,params.max_depth,metrics.mae_minutes
0,first-tree,4,3.465814


In [8]:
depth_values = [1, 2, 3, 5, 8, 12]

for depth in depth_values:

    # Each loop iteration is recorded as a separate run
    with mlflow.start_run(run_name="tree-depth-" + str(depth)):

        model = DecisionTreeRegressor(max_depth=depth, random_state=42)
        model.fit(X_train, y_train)
        mae = mean_absolute_error(y_test, model.predict(X_test))

        mlflow.log_param("max_depth", depth)
        mlflow.log_metric("mae_minutes", mae)

    print("max_depth =", depth, "   MAE =", round(mae, 2))

max_depth = 1    MAE = 6.11
max_depth = 2    MAE = 4.59
max_depth = 3    MAE = 4.04
max_depth = 5    MAE = 3.31
max_depth = 8    MAE = 3.96
max_depth = 12    MAE = 4.39


In [9]:
best = mlflow.search_runs(
    experiment_names=["P06-delivery"],
    filter_string="attributes.status = 'FINISHED'",   # only successful runs
    order_by=["metrics.mae_minutes ASC"],              # lowest MAE first
    max_results=1,                                     # only the best run
)

# The first (and only) row is the best run
best_run_name = best["tags.mlflow.runName"][0]
best_depth    = best["params.max_depth"][0]
best_mae      = best["metrics.mae_minutes"][0]

print("Best run  :", best_run_name)
print("Best depth:", best_depth)
print("Best MAE  :", round(best_mae, 2))

Best run  : tree-depth-5
Best depth: 5
Best MAE  : 3.31


In [10]:
# Parameters are stored as text, so convert the depth to an integer
best_depth = int(best_depth)

with mlflow.start_run(run_name="winner-saved"):

    # Retrain the model using the best depth
    model = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
    model.fit(X_train, y_train)
    mae = mean_absolute_error(y_test, model.predict(X_test))

    mlflow.log_param("max_depth", best_depth)
    mlflow.log_metric("mae_minutes", mae)

    # Save the trained model as an artifact and register it
    mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        registered_model_name="delivery-time-model",
        serialization_format="cloudpickle",
    )

print("Model saved and registered. MAE =", round(mae, 2))

2026/11/02 23:55:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model saved and registered. MAE = 3.31


Successfully registered model 'delivery-time-model'.
Created version '1' of model 'delivery-time-model'.


In [11]:
client = MlflowClient()

versions = client.search_model_versions("name='delivery-time-model'")

for v in versions:
    print("Version:", v.version, "   Created by run:", v.run_id)

Version: 1    Created by run: e2534123da1a4ec8bccf4e228c8c3d60


In [12]:
# Load version 1 of the registered model
loaded_model = mlflow.sklearn.load_model("models:/delivery-time-model/1")

# A new order: 8 km away, 3 items, placed at 7 pm (hour 19)
new_order = pd.DataFrame({
    "distance_km": [8],
    "items":       [3],
    "hour":        [19],
})

predicted_time = loaded_model.predict(new_order)[0]
print("Predicted delivery time:", round(predicted_time, 1), "minutes")

Predicted delivery time: 42.4 minutes


In [13]:
import subprocess
import sys

# Start the MLflow web interface as a background process
ui_process = subprocess.Popen([
    sys.executable, "-m", "mlflow", "ui",
    "--backend-store-uri", "sqlite:///mlflow.db",
    "--port", "5001",
])

print("MLflow UI is starting. Open http://127.0.0.1:5001 in your browser.")

MLflow UI is starting. Open http://127.0.0.1:5001 in your browser.
